# 05. Clustering

### Objective
Assign each 30-second window to a player archetype using K-Means, then compute soft IDW membership scores and temporal deltas.

### I/O
- **Reads**: `../../data/processed/4_activity_contributions.csv`
- **Writes**: `../../data/processed/5_clustered_telemetry.csv`
- **Writes**: `../../data/processed/cluster_centroids.json`

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import json, os

INPUT_PATH  = '../../data/processed/4_activity_contributions.csv'
OUTPUT_PATH = '../../data/processed/5_clustered_telemetry.csv'
MODEL_DIR   = '../../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Libraries imported')

Libraries imported


In [2]:
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df)} rows')

Loaded 3240 rows


Two experiments shaped the clustering decisions made below.

**Experiment A** (`experiments/experiment_A_feature_space_validation/Experiment_A_Complete.ipynb`)
compared clustering on raw normalized features vs. activity percentages (`pct_combat`, `pct_collect`, `pct_explore`)
at K=2,3,4,5 using silhouette, Davies-Bouldin, and Calinski-Harabasz scores.

The actual output showed percentage features produce **higher** silhouette scores at every K value:

| K | pct silhouette | raw silhouette |
|---|---|---|
| 2 | 0.5799 | 0.5058 |
| 3 | **0.4948** | 0.4635 |
| 4 | 0.4856 | 0.3749 |
| 5 | 0.4798 | 0.3117 |

The experiment does confirm the compositional constraint mathematically (pct sums to 1.0),
but empirically the pct space clusters more cleanly because the archetypes ARE defined in
percentage space by design - a Combat player is one with high pct_combat. Raw features add
11-dimensional noise where the archetype signal is diluted. The pct approach is both
practically interpretable (centroids read as "65% combat, 5% collect") and empirically optimal.

**Experiment B** (`experiments/experiment_B_clustering_config/Experiment_B_Complete.ipynb`)
ran a 108-config grid search for raw-feature clustering. Its best config was K=3, 8 features,
minmax_log_sparse, no outlier cap (silhouette=0.3928) - still lower than pct K=3 (0.4948).
The K=3 decision is confirmed: K=2 merges Collect+Explore into one blob, K=4 over-segments.

Because of Experiment A's result, pct features are used below - both interpretable and empirically better separated.

In [3]:
features = ['pct_combat', 'pct_collect', 'pct_explore']
X = df[features].fillna(0)

print(f'Clustering features: {features}')
print(f'Shape: {X.shape}')

Clustering features: ['pct_combat', 'pct_collect', 'pct_explore']
Shape: (3240, 3)


In [4]:
print('Running K-Means (K=3)...')
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)

print('Clustering complete')
print(f'Cluster centers:\n{kmeans.cluster_centers_}')

Running K-Means (K=3)...
Clustering complete
Cluster centers:
[[0.65845085 0.04413931 0.29740984]
 [0.00998018 0.14233362 0.8476862 ]
 [0.32672967 0.18185444 0.49141589]]


In [5]:
# Map each K-Means cluster to the nearest named archetype using the Hungarian algorithm.
# This gives a stable, deterministic assignment regardless of which cluster index
# K-Means happens to assign on any given run.
ideal_centers = np.array([
    [1.0, 0.0, 0.0],  # ideal Combat
    [0.0, 1.0, 0.0],  # ideal Collection
    [0.0, 0.0, 1.0],  # ideal Exploration
])

distances = cdist(kmeans.cluster_centers_, ideal_centers, metric='euclidean')
row_ind, col_ind = linear_sum_assignment(distances)

archetype_names = ['Combat', 'Collection', 'Exploration']
mapping = {row_ind[i]: archetype_names[col_ind[i]] for i in range(len(row_ind))}

df['archetype'] = df['cluster'].map(mapping)
print(f'Cluster mapping: {mapping}')
print(f'Archetype distribution:\n{df["archetype"].value_counts()}')

Cluster mapping: {np.int64(0): 'Combat', np.int64(1): 'Exploration', np.int64(2): 'Collection'}
Archetype distribution:
archetype
Exploration    1507
Collection      989
Combat          744
Name: count, dtype: int64


## Soft Membership Computation

Compute a fractional membership score for each archetype using Inverse Distance Weighting (IDW).
Each window gets a probability-like vector such as `[combat=0.6, collect=0.3, explore=0.1]`
instead of a binary one-hot label. This prevents hard jumps in the difficulty signal when a
player crosses an archetype boundary.

Experiment B Section 2 (`experiments/experiment_B_clustering_config/Experiment_B_Complete.ipynb`)
quantifies the smoothness improvement: soft IDW reduces mean transition magnitude by ~34%
compared to hard one-hot labels.

In [6]:
print('Computing soft membership...')

# Distance from each window to each cluster centroid
distances = kmeans.transform(X)

# Invert: closer centroid = higher score
inv_distances = 1 / (distances + 1e-10)

# Normalize so all three scores sum to 1.0 per window
soft_membership = inv_distances / inv_distances.sum(axis=1, keepdims=True)

combat_idx  = [k for k, v in mapping.items() if v == 'Combat'][0]
collect_idx = [k for k, v in mapping.items() if v == 'Collection'][0]
explore_idx = [k for k, v in mapping.items() if v == 'Exploration'][0]

df['soft_combat']  = soft_membership[:, combat_idx]
df['soft_collect'] = soft_membership[:, collect_idx]
df['soft_explore'] = soft_membership[:, explore_idx]

print('Soft membership computed')
print(f'Mean soft_combat:  {df["soft_combat"].mean():.4f}')
print(f'Mean soft_collect: {df["soft_collect"].mean():.4f}')
print(f'Mean soft_explore: {df["soft_explore"].mean():.4f}')

Computing soft membership...
Soft membership computed
Mean soft_combat:  0.2701
Mean soft_collect: 0.3130
Mean soft_explore: 0.4169


## Temporal Delta Computation

Calculate window-to-window change in soft membership scores per player.
A positive delta_combat means the player is becoming more combat-focused this window.
A negative delta means they are shifting away. These deltas are the primary variance
driver in the ANFIS target variable - see notebook 06 and Experiment C for why.

In [7]:
print('Computing temporal deltas...')

# Sort within each player's timeline before differencing.
# groupby('userId') ensures we never subtract User B's first window from User A's last.
df_sorted = df.sort_values(['userId', 'timestamp'])

df_sorted['delta_combat']  = df_sorted.groupby('userId')['soft_combat'].diff().fillna(0)
df_sorted['delta_collect'] = df_sorted.groupby('userId')['soft_collect'].diff().fillna(0)
df_sorted['delta_explore'] = df_sorted.groupby('userId')['soft_explore'].diff().fillna(0)

df = df_sorted.copy()

print('Deltas computed (first window per player = 0)')
print(f'delta_combat range:  [{df["delta_combat"].min():.3f}, {df["delta_combat"].max():.3f}]')
print(f'delta_collect range: [{df["delta_collect"].min():.3f}, {df["delta_collect"].max():.3f}]')
print(f'delta_explore range: [{df["delta_explore"].min():.3f}, {df["delta_explore"].max():.3f}]')

Computing temporal deltas...
Deltas computed (first window per player = 0)
delta_combat range:  [-0.879, 0.833]
delta_collect range: [-0.866, 0.960]
delta_explore range: [-0.955, 0.910]


In [8]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')
print('Clustering + soft membership + deltas complete!')

Saved to ../../data/processed/5_clustered_telemetry.csv
Clustering + soft membership + deltas complete!


In [9]:
# Export cluster centroids for the demo UI.
# The TypeScript engine uses these to recompute IDW membership for live player input.
CENTROIDS_OUTPUT = os.path.join(MODEL_DIR, 'cluster_centroids.json')
CENTROIDS_DEPLOY = '../../../anfis-demo-ui/models/cluster_centroids.json'

centroids_export = {}
for cluster_id, archetype_name in mapping.items():
    center = kmeans.cluster_centers_[cluster_id]
    centroids_export[str(cluster_id)] = {
        'archetype': archetype_name,
        'centroid': {
            'pct_combat':  float(center[0]),
            'pct_collect': float(center[1]),
            'pct_explore': float(center[2])
        }
    }

with open(CENTROIDS_OUTPUT, 'w') as f:
    json.dump(centroids_export, f, indent=2)

try:
    os.makedirs(os.path.dirname(CENTROIDS_DEPLOY), exist_ok=True)
    with open(CENTROIDS_DEPLOY, 'w') as f:
        json.dump(centroids_export, f, indent=2)
    print(f'Deployed to: {CENTROIDS_DEPLOY}')
except Exception:
    print('Deploy path not found - skipping deploy copy')

print(f'Saved to: {CENTROIDS_OUTPUT}')
print(json.dumps(centroids_export, indent=2))

Deployed to: ../../../anfis-demo-ui/models/cluster_centroids.json
Saved to: ../../data/models\cluster_centroids.json
{
  "0": {
    "archetype": "Combat",
    "centroid": {
      "pct_combat": 0.6584508458557516,
      "pct_collect": 0.04413931032430267,
      "pct_explore": 0.2974098438199457
    }
  },
  "1": {
    "archetype": "Exploration",
    "centroid": {
      "pct_combat": 0.009980181516020203,
      "pct_collect": 0.14233362069180278,
      "pct_explore": 0.8476861977921768
    }
  },
  "2": {
    "archetype": "Collection",
    "centroid": {
      "pct_combat": 0.3267296749125025,
      "pct_collect": 0.18185443733745807,
      "pct_explore": 0.4914158877500394
    }
  }
}
